In [ ]:
# bi_service_refresh — rebuild the BI mart after gold is written, then refresh the
# Direct Lake model. Runs as the LAST activity of Pipeline_eod_sale_service (after the
# transform+DQ notebook), so the dashboard reflects only DQ-passed data.
# Full rebuild each run: the mart is a small projection, CREATE OR REPLACE is atomic.


In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql("""CREATE OR REPLACE TABLE gold.bi_eod_sale_service AS
SELECT report_date, service_name, provider_id, order_type, store_id, store_name,
       staff_id, payment_method, transaction_id, product_code, product_name, quantity,
       CAST(total_amount AS BIGINT) AS total_amount,
       CAST(amount_before_vat AS BIGINT) AS amount_before_vat,
       CAST(amount_vat AS BIGINT) AS amount_vat,
       CAST(total_commission AS BIGINT) AS total_commission,
       CAST(commission_on_vnd AS BIGINT) AS commission_on_vnd,
       CAST(purchase_price_with_tax AS BIGINT) AS purchase_price_with_tax,
       CAST(purchase_price_without_tax AS BIGINT) AS purchase_price_without_tax
FROM gold.fact_eod_sale_service""")
print("mart gold.bi_eod_sale_service rows:", spark.read.table("gold.bi_eod_sale_service").count())


In [ ]:
# Refresh the Direct Lake model so the dashboard is current (Direct Lake usually
# reframes on its own, but an explicit refresh guarantees it). Best-effort: a missing
# model or permission issue must not fail the pipeline.
try:
    import sempy.fabric as fabric
    fabric.refresh_dataset("Sales Service", refresh_type="full")
    print("semantic model 'Sales Service' refreshed")
except Exception as e:
    print("model refresh skipped:", str(e)[:200])
